# Expense categoriser: train and test

This notebook trains a model to put an expense into one of 12 categories, then tests it on data it has not seen.

**Inputs** (produced earlier)
- `outputs/personal_transactions_expenses.csv` and `outputs/khalifa_transactions_expenses.csv`, from the cleaning notebook
- `merchant_labels.xlsx`, the merchant labelling sheet (`final_category` column, plus optional `row_override` on the *Ambiguous rows* sheet)

**What the model sees:** only what a new SMS would give it — the raw merchant text as the bank wrote it, and the amount. (Location and bank were tried and dropped: they added noise and hurt the external test.) It never sees the cleaned merchant name, because that name was made with a brand table and would let the model cheat.

**How it is tested** (five splits, from most to least optimistic):
1. **Random (5-fold).** The usual approach. Optimistic, because the same merchants appear in train and test.
2. **Unseen merchants (5-fold, grouped).** No merchant brand in a test fold appears in its training folds; every merchant is tested once. The honest score for a brand-new merchant.
3. **Unseen person.** Train on two people, test on the third (rotated).
4. **Future.** Train on transactions before 2025, test on 2025.
5. **External (Khalifa University).** Train on all personal data, test on the ADIB dataset: a different bank, different emirate (Abu Dhabi) and different years.

**Models:** a majority-class baseline, a merchant lookup table, and TF-IDF character n-grams with logistic regression. An optional sentence-embedding model can be switched on at the end.

> Labels are provisional until reviewed: most were suggested by brand/keyword rules or by Claude. Re-run this notebook after reviewing `merchant_labels.xlsx`. Scores against unreviewed labels partly measure how well the model re-learns those rules.

## 1. Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Masters of Applied AI/MAAI902/sms_expense_project_bundle"
!ls

In [ ]:
from pathlib import Path

TRANSACTIONS_DIR = Path('outputs')
LABELS_FILE = Path('merchant_labels.xlsx')
MODEL_DIR = Path('models')

TEMPORAL_CUTOFF = '2025-01-01'   # train before, test from this date (personal data only)
RANDOM_STATE = 42
RUN_EMBEDDINGS = False           # True = also try a sentence-embedding model (needs: pip install sentence-transformers)
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

## 2. Load expenses and attach labels

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 200)

personal = pd.read_csv(TRANSACTIONS_DIR / 'personal_transactions_expenses.csv', parse_dates=['date'])
khalifa = pd.read_csv(TRANSACTIONS_DIR / 'khalifa_transactions_expenses.csv', parse_dates=['date'])
personal['dataset'], khalifa['dataset'] = 'personal', 'khalifa'

merchants = pd.read_excel(LABELS_FILE, sheet_name='Merchants')
categories = pd.read_excel(LABELS_FILE, sheet_name='Categories').category
CATS = [c for c in categories if c != 'Total']
label_map = dict(zip(merchants.merchant, merchants.final_category))
source_map = dict(zip(merchants.merchant, merchants.source))

overrides = pd.read_excel(LABELS_FILE, sheet_name='Ambiguous rows')
overrides = dict(zip(overrides.txn_id, overrides.row_override.where(overrides.row_override.notna())))
overrides = {k: v for k, v in overrides.items() if isinstance(v, str)}

def attach_labels(df):
    df = df.copy()
    df['label'] = df.merchant_clean.map(label_map)
    df['label_source'] = df.merchant_clean.map(source_map)
    has_override = df.txn_id.isin(overrides)
    df.loc[has_override, 'label'] = df.loc[has_override, 'txn_id'].map(overrides)
    df.loc[has_override, 'label_source'] = 'row_override'
    return df

personal, khalifa = attach_labels(personal), attach_labels(khalifa)

for name, d in [('personal', personal), ('khalifa', khalifa)]:
    missing = d.label.isna().sum()
    assert missing == 0, f'{missing} {name} expenses have no label - add their merchants to merchant_labels.xlsx'
    bad = set(d.label) - set(CATS)
    assert not bad, f'Unknown category names in labels: {bad}'

print(f'{len(personal):,} personal and {len(khalifa):,} Khalifa expenses, all labelled; {len(overrides)} row overrides')
pd.concat([personal.label.value_counts().rename('personal'), khalifa.label.value_counts().rename('khalifa')], axis=1).fillna(0).astype(int)

In [ ]:
# Where the labels came from (matters when interpreting scores)
both = pd.concat([personal, khalifa], ignore_index=True)
pd.crosstab(both.label_source, both.dataset, margins=True)

## 3. Features

`text` is the merchant as the bank wrote it, with store numbers and phone numbers removed. Character n-grams (2–5 letters) cope with truncated and misspelt names such as `SMART DUBAI GOVE` or `RESTURANT`. Log-amount is added as a small side feature (a AED 4 charge and a AED 4,000 charge rarely share a category).

In [ ]:
def make_text(df):
    t = df.merchant.fillna('').astype(str).str.upper()
    t = t.str.replace(r'\+?\d[\d -]{4,}', ' ', regex=True)      # phone / reference numbers
    t = t.str.replace(r'[^A-Z& ]', ' ', regex=True)               # keep letters only
    return t.str.replace(r'\s+', ' ', regex=True).str.strip()

for d in (personal, khalifa):
    d['text'] = make_text(d)
    d['log_amount'] = np.log1p(d.amount_aed.fillna(d.amount).clip(lower=0))

FEATURES = ['text', 'log_amount']
personal[['merchant', 'text', 'bank', 'amount_aed', 'label']].sample(6, random_state=1)

## 4. Models

In [ ]:
class MajorityClass:
    name = 'Majority class'
    def fit(self, X, y):
        self.label_ = pd.Series(y).mode()[0]; self.classes_ = np.array(CATS); return self
    def predict(self, X):
        return np.full(len(X), self.label_)
    def predict_proba(self, X):
        p = np.zeros((len(X), len(CATS))); p[:, CATS.index(self.label_)] = 1; return p

class MerchantLookup:
    # Memorise the most common label per exact merchant text; unseen merchants get the majority class.
    name = 'Merchant lookup'
    def fit(self, X, y):
        d = pd.DataFrame({'t': X.text.values, 'y': np.asarray(y)})
        self.table_ = d.groupby('t').y.agg(lambda s: s.mode()[0]).to_dict()
        self.fallback_ = d.y.mode()[0]; self.classes_ = np.array(CATS); return self
    def predict(self, X):
        return np.array([self.table_.get(t, self.fallback_) for t in X.text])
    def predict_proba(self, X):
        p = np.zeros((len(X), len(CATS)))
        for i, t in enumerate(X.text):
            p[i, CATS.index(self.table_.get(t, self.fallback_))] = 1.0 if t in self.table_ else 0.0
        return p

def tfidf_logreg():
    text = FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=1, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ])
    pre = ColumnTransformer([
        ('text', text, 'text'),
        ('amount', StandardScaler(), ['log_amount']),
    ])
    m = Pipeline([('features', pre),
                  ('clf', LogisticRegression(max_iter=3000, C=10, class_weight='balanced'))])
    m.name = 'TF-IDF + logistic regression'
    return m

MODELS = [MajorityClass, MerchantLookup, tfidf_logreg]

## 5. The five test set-ups

In [ ]:
def split_random(df):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    return [('random (5-fold)', df.iloc[tr], df.iloc[te]) for tr, te in skf.split(df, df.label)]

def split_unseen_merchant(df):
    gkf = GroupKFold(n_splits=5)
    return [('unseen merchants (5-fold)', df.iloc[tr], df.iloc[te]) for tr, te in gkf.split(df, groups=df.merchant_clean)]

def split_unseen_person(df):
    return [(f'unseen person: {u}', df[df.user != u], df[df.user == u]) for u in sorted(df.user.unique())]

def split_future(df):
    return [(f'future (from {TEMPORAL_CUTOFF})', df[df.date < TEMPORAL_CUTOFF], df[df.date >= TEMPORAL_CUTOFF])]

def split_external(df):
    return [('external: Khalifa ADIB', df, khalifa)]

SPLITS = [split_random, split_unseen_merchant, split_unseen_person, split_future, split_external]

def evaluate(model_factory, train, test):
    m = model_factory()
    m.fit(train[FEATURES], train.label)
    pred = m.predict(test[FEATURES])
    proba = m.predict_proba(test[FEATURES])
    seen = test.merchant_clean.isin(set(train.merchant_clean))
    return m, pred, proba, dict(
        accuracy=accuracy_score(test.label, pred),
        macro_f1=f1_score(test.label, pred, average='macro', labels=CATS, zero_division=0),
        weighted_f1=f1_score(test.label, pred, average='weighted', zero_division=0),
        n_train=len(train), n_test=len(test), test_merchants_seen_in_train=seen.mean())

results, keep = [], {}
for split_fn in SPLITS:
    for split_name, tr, te in split_fn(personal):
        for factory in MODELS:
            m, pred, proba, r = evaluate(factory, tr, te)
            results.append(dict(split=split_name, model=m.name, **r))
            # keep out-of-fold predictions; folds of the same set-up are stacked together
            prev = keep.get((split_name, m.name))
            keep[(split_name, m.name)] = (te, pred, proba, m) if prev is None else (
                pd.concat([prev[0], te]), np.concatenate([prev[1], pred]), np.vstack([prev[2], proba]), m)

# average the 5 folds of the k-fold set-ups
res = (pd.DataFrame(results).groupby(['split', 'model'], sort=False)
         .agg(accuracy=('accuracy', 'mean'), macro_f1=('macro_f1', 'mean'), weighted_f1=('weighted_f1', 'mean'),
              n_train=('n_train', 'mean'), n_test=('n_test', 'sum'),
              test_merchants_seen_in_train=('test_merchants_seen_in_train', 'mean')).reset_index())
res['n_train'] = res.n_train.round().astype(int)
show = res.copy()
for c in ['accuracy', 'macro_f1', 'weighted_f1', 'test_merchants_seen_in_train']:
    show[c] = show[c].map('{:.1%}'.format)
show

In [ ]:
# One line per test set-up (unseen-person averaged over the three people), best model highlighted
res['setup'] = res.split.str.replace(r'unseen person: .*', 'unseen person (avg)', regex=True)
summary = res.pivot_table(index='setup', columns='model', values='macro_f1', aggfunc='mean')
order = ['random (5-fold)', 'unseen merchants (5-fold)', 'unseen person (avg)', f'future (from {TEMPORAL_CUTOFF})', 'external: Khalifa ADIB']
summary = summary.loc[order, ['Majority class', 'Merchant lookup', 'TF-IDF + logistic regression']]
print('Macro-F1 by test set-up:')
summary.style.format('{:.1%}').highlight_max(axis=1, color='#d9ead3')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
summary.plot.bar(ax=ax, width=0.8, color=['#bbbbbb', '#8fb3d9', '#2b6cb0'])
ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1); ax.set_xlabel('')
ax.set_title('Macro-F1 drops as the test data gets less like the training data')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.xticks(rotation=15, ha='right'); ax.legend(frameon=False, fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Where it goes wrong: unseen merchants

These are out-of-fold predictions: every personal expense, predicted by a model that never saw its merchant.

In [ ]:
te, pred, proba, model = keep[('unseen merchants (5-fold)', 'TF-IDF + logistic regression')]
print(classification_report(te.label, pred, labels=CATS, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(te.label, pred, labels=CATS, normalize='true'),
                       display_labels=CATS).plot(ax=ax, cmap='Blues', values_format='.0%', colorbar=False)
ax.set_title('Unseen merchants: share of each true class (rows) predicted as each class')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Most frequent mistakes, with the merchant text the model saw
errors = te.assign(pred=pred)[lambda d: d.label != d.pred]
(errors.groupby(['label', 'pred']).agg(n=('text', 'size'), examples=('text', lambda s: ', '.join(s.unique()[:4])))
       .sort_values('n', ascending=False).head(12))

## 7. Automatic filing vs accuracy

In use, the model files an expense automatically only when it is confident, and sends the rest for a quick human check. The table shows that trade-off for new merchants and for the external dataset.

In [ ]:
def coverage_table(key, thresholds=(0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)):
    te, pred, proba, m = keep[key]
    conf = proba.max(axis=1)
    rows = []
    for t in thresholds:
        auto = conf >= t
        rows.append(dict(threshold=t, auto_filed=auto.mean(),
                         accuracy_on_auto=(pred[auto] == te.label.values[auto]).mean() if auto.any() else np.nan))
    return pd.DataFrame(rows)

cov = {name: coverage_table((name, 'TF-IDF + logistic regression')) for name in ['unseen merchants (5-fold)', 'external: Khalifa ADIB']}
out = pd.concat(cov, axis=1)
out.style.format('{:.0%}', na_rep='-').format('{:.1f}', subset=[(n, 'threshold') for n in cov])

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, t in cov.items():
    ax.plot(t.auto_filed, t.accuracy_on_auto, marker='o', label=name)
ax.set_xlabel('Share of expenses filed automatically'); ax.set_ylabel('Accuracy on those')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title('Confidence threshold trade-off'); ax.grid(alpha=0.3); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 8. Optional: sentence-embedding model

In [ ]:
if RUN_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print('pip install sentence-transformers to run this cell'); RUN_EMBEDDINGS = False

if RUN_EMBEDDINGS:
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    cache = {}
    def embed(texts):
        new = [t for t in set(texts) if t not in cache]
        if new:
            for t, v in zip(new, encoder.encode(new, batch_size=64, show_progress_bar=False)):
                cache[t] = v
        return np.vstack([cache[t] for t in texts])

    class EmbeddingLogReg:
        name = 'Embeddings + logistic regression'
        def fit(self, X, y):
            self.clf = LogisticRegression(max_iter=3000, class_weight='balanced').fit(embed(list(X.text)), y)
            self.classes_ = self.clf.classes_; return self
        def predict(self, X): return self.clf.predict(embed(list(X.text)))
        def predict_proba(self, X): return self.clf.predict_proba(embed(list(X.text)))

    extra = []
    for split_fn in [split_unseen_merchant, split_external]:
        for split_name, tr, te in split_fn(personal):
            m, pred, proba, r = evaluate(EmbeddingLogReg, tr, te)
            extra.append(dict(split=split_name, model=m.name, **r))
    display(pd.DataFrame(extra).groupby(['split', 'model'], sort=False)[['accuracy', 'macro_f1']].mean())
else:
    print('Skipped (set RUN_EMBEDDINGS = True to run).')

## 9. Train the final model on everything and save it

In [ ]:
final = tfidf_logreg()
final.fit(personal[FEATURES], personal.label)
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump({'model': final, 'categories': CATS, 'features': FEATURES}, MODEL_DIR / 'expense_categoriser.joblib')
res.to_csv(MODEL_DIR / 'results.csv', index=False)
print('Saved', MODEL_DIR / 'expense_categoriser.joblib')

def categorise(merchant_texts, amount_aed=50.0):
    # Quick demo: categorise merchant strings as they appear in an SMS.
    d = pd.DataFrame({'merchant': merchant_texts, 'location': ''})
    X = pd.DataFrame({'text': make_text(d), 'log_amount': np.log1p(amount_aed)})
    p = final.predict_proba(X)
    return pd.DataFrame({'merchant': merchant_texts, 'category': final.classes_[p.argmax(1)],
                         'confidence': p.max(1).round(2)})

# Brands it has never seen (e.g. SHELL) are where it struggles: nothing in the letters says 'fuel'.
categorise(['KARAK HOUSE JLT', 'AL MAYA SUPERMARKET', 'DUBAI POLICE FINES', 'NETFLIX.COM', 'ASTER PHARMACY 112',
            'CAREEM FOOD', 'SHELL DUBAI', 'CINEMA CITY AL ZAHIA', 'FLYDUBAI', 'ICP SMART SERVICES', 'GEMS WELLINGTON SCHOOL'])

## 10. Categorising new merchants: cascade with confidence scores

A new SMS goes through up to four tiers, cheapest and most precise first. Each tier gives a score for **every** category. The first tier whose top score clears `REVIEW_THRESHOLD` decides. If none does, the best guess is still returned, tagged `needs_review = True` for the user.

| Tier | What it uses | Confidence |
|---|---|---|
| 1. lookup | merchant already in the label table | label quality: Danish-confirmed 0.99, high 0.97, medium 0.90, low/unresolved 0.50, lowered if that text has mixed labels (Amazon, Noon...) |
| 2. rules | `scripts/rules.py` brands and keywords | fixed: manual 0.95, brand 0.92 (ambiguous brand 0.70), keyword 0.80 |
| 3. model | TF-IDF + logistic regression, **calibrated on held-out merchants** | calibrated probability |
| 4. LLM (optional) | plug in any function that returns category scores | whatever it returns |

Calibration matters here. Raw logistic-regression scores are over-confident on unseen merchants. The calibrated scores are fitted with grouped (unseen-merchant) folds, so a score of 0.8 should mean "right about 80% of the time on a brand it hasn't seen".

In [ ]:
import sys
sys.path.insert(0, 'scripts')
import rules
from sklearn.calibration import CalibratedClassifierCV

TARGET_AUTO_ACCURACY = 0.90   # auto-filed new merchants should be right at least this often
REVIEW_THRESHOLD = None       # None = choose from data (below); or set e.g. 0.7

def calibrated_model(train):
    folds = list(GroupKFold(n_splits=5).split(train, groups=train.merchant_clean))
    m = CalibratedClassifierCV(tfidf_logreg(), method='sigmoid', cv=folds)
    return m.fit(train[FEATURES], train.label)

def proba_in_cat_order(m, X):
    p = m.predict_proba(X)
    order = [list(m.classes_).index(c) for c in CATS]
    return p[:, order]

# Out-of-fold calibrated scores on UNSEEN merchants: the worst case, used to pick the threshold.
oof = np.zeros((len(personal), len(CATS)))
for tr, te in GroupKFold(n_splits=5).split(personal, groups=personal.merchant_clean):
    m = calibrated_model(personal.iloc[tr])
    oof[te] = proba_in_cat_order(m, personal.iloc[te][FEATURES])
oof_conf = oof.max(1)
oof_right = np.array(CATS)[oof.argmax(1)] == personal.label.values

rows = []
for t in np.round(np.arange(0.3, 0.96, 0.05), 2):
    auto = oof_conf >= t
    rows.append(dict(threshold=t, auto_filed=auto.mean(), accuracy_on_auto=oof_right[auto].mean() if auto.any() else np.nan,
                     sent_to_review=(~auto).mean()))
thr_table = pd.DataFrame(rows)
if REVIEW_THRESHOLD is None:
    ok = thr_table[thr_table.accuracy_on_auto >= TARGET_AUTO_ACCURACY]
    REVIEW_THRESHOLD = float(ok.threshold.min()) if len(ok) else 0.9
print(f'REVIEW_THRESHOLD = {REVIEW_THRESHOLD}  (unseen-merchant worst case, target {TARGET_AUTO_ACCURACY:.0%} accuracy on auto-filed)')
thr_table.style.format({'auto_filed': '{:.0%}', 'accuracy_on_auto': '{:.0%}', 'sent_to_review': '{:.0%}'})

In [ ]:
# Reliability check: does a score of x really mean x% right? (unseen merchants)
bins = pd.cut(oof_conf, [0, .3, .4, .5, .6, .7, .8, .9, 1.0])
rel = pd.DataFrame({'bin': bins, 'conf': oof_conf, 'right': oof_right}).groupby('bin', observed=True).agg(
    n=('right', 'size'), mean_confidence=('conf', 'mean'), actual_accuracy=('right', 'mean'))
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1, label='perfect calibration')
ax.plot(rel.mean_confidence, rel.actual_accuracy, marker='o', label='calibrated model')
ax.set_xlabel('Predicted confidence'); ax.set_ylabel('Actual accuracy'); ax.set_title('Reliability on unseen merchants')
ax.legend(frameon=False); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
rel.round(2)

In [ ]:
# Final calibrated model on all personal data + the lookup table
cal_final = calibrated_model(personal)

def merchant_confidence(r):
    if r.get('label_status') == 'confirmed' or str(r.get('source', '')).startswith('danish'):
        return 0.99
    if r.get('label_status') == 'unresolved':
        return 0.50
    return {'high': 0.97, 'medium': 0.90, 'low': 0.50}.get(r.get('confidence'), 0.80)

m_conf = {r['merchant']: merchant_confidence(r) for r in merchants.to_dict('records')}
known = pd.concat([personal, khalifa])[['text', 'merchant_clean', 'label']]
known['conf'] = known.merchant_clean.map(m_conf).fillna(0.8)
LOOKUP = {}
for t, g in known.groupby('text'):
    shares = g.label.value_counts(normalize=True)
    LOOKUP[t] = (shares.index[0], float(shares.iloc[0] * g.conf.mean()))   # mixed labels -> lower confidence

RULE_CONF = {'manual': 0.95, 'brand': 0.92, 'keyword': 0.80, 'rule': 0.30}
LLM_CATEGORISER = None   # optional: f(merchant_text, amount_aed) -> {category: score}; see the note below

def _spread(cat, conf, base):
    # Put `conf` on cat and share the rest in proportion to the model's scores, so every category gets a score.
    other = base.copy(); other[CATS.index(cat)] = 0
    other = other / other.sum() * (1 - conf) if other.sum() > 0 else np.full(len(CATS), (1 - conf) / (len(CATS) - 1))
    other[CATS.index(cat)] = conf
    return other

def categorise_new(merchant_texts, amounts_aed=50.0, threshold=None, show_scores=True):
    threshold = REVIEW_THRESHOLD if threshold is None else threshold
    d = pd.DataFrame({'merchant': list(merchant_texts), 'location': ''})
    d['amount_aed'] = amounts_aed
    d['text'] = make_text(d); d['log_amount'] = np.log1p(d.amount_aed.clip(lower=0))
    model_p = proba_in_cat_order(cal_final, d[FEATURES])
    out = []
    for i, r in d.iterrows():
        tiers = []
        if r.text in LOOKUP:
            cat, c = LOOKUP[r.text]; tiers.append(('lookup', _spread(cat, c, model_p[i])))
        cat, src, rc = rules.suggest(r.text, r.merchant)
        if cat:
            c = RULE_CONF.get(src, 0.8) if not (src == 'brand' and rc == 'medium') else 0.70
            tiers.append(('rules', _spread(cat, c, model_p[i])))
        tiers.append(('model', model_p[i]))
        if LLM_CATEGORISER is not None:
            s = LLM_CATEGORISER(r.merchant, r.amount_aed)
            tiers.append(('llm', np.array([s.get(c, 0.0) for c in CATS]) / max(sum(s.values()), 1e-9)))
        chosen = next(((n, p) for n, p in tiers if p.max() >= threshold), None)
        review = chosen is None
        if review:
            chosen = max(tiers, key=lambda t: t[1].max())   # best guess anyway
        name, p = chosen
        top = np.argsort(p)[::-1]
        row = dict(merchant=r.merchant, category=CATS[top[0]], confidence=round(float(p[top[0]]), 2),
                   decided_by=name, needs_review=review,
                   runner_up=f'{CATS[top[1]]} ({p[top[1]]:.2f})')
        if show_scores:
            row.update({c: round(float(v), 2) for c, v in zip(CATS, p)})
        out.append(row)
    return pd.DataFrame(out)

categorise_new(['NEW SALAH ALDIN-1076', 'KARAK HOUSE JLT', 'ADNOC 1234', 'SHELL DUBAI', 'CARREFOUR MOE',
                'NETFLIX.COM', 'ICP SMART SERVICES', 'XYZ HOLDINGS LLC', 'AL NOOR TRADING EST'],
               amounts_aed=[40, 12, 150, 120, 230, 56, 300, 500, 80])

In [ ]:
# Save everything needed to categorise later without retraining
joblib.dump({'model': cal_final, 'categories': CATS, 'features': FEATURES, 'lookup': LOOKUP,
             'rule_conf': RULE_CONF, 'review_threshold': REVIEW_THRESHOLD},
            MODEL_DIR / 'expense_categoriser_cascade.joblib')
print('Saved', MODEL_DIR / 'expense_categoriser_cascade.joblib')

**Plugging in an LLM (tier 4).** Set `LLM_CATEGORISER` to any function that takes the merchant text and amount and returns a score per category, e.g. by asking a model to answer in JSON like `{"Transport_Auto": 0.85, "Shopping_Retail": 0.1, ...}` using the 12 category descriptions from the Categories sheet. Re-run `categorise_new`. Only merchants that the lookup, rules and model are unsure about reach it, which keeps cost low.

**Closing the loop.** When the user fixes a `needs_review` item, add the merchant to `merchant_labels.xlsx` (source `danish`). It then goes straight to the lookup tier with confidence 0.99 and is never asked about again.

## Notes for the write-up

- **Report the unseen-merchant, unseen-person and external numbers, not the random split.** The gap between them is a finding in itself: it shows how much published random-split results overstate real-world accuracy.
- The **merchant lookup** baseline is strong whenever merchants repeat and collapses on unseen ones. Beating it on the unseen splits is what shows the model generalises.
- **Unseen merchants is the hard case (about 50% macro-F1).** Character n-grams can't know that `MCDONALDS` or `SHELL` is food or fuel without having seen them. This is where pretrained embeddings or an LLM with world knowledge should help, and it's the most interesting comparison to run next (section 8).
- **Utilities_Telecom**, **Digital_Subscriptions** and **Government_Fees** have few distinct merchants (a handful of brands carry most transactions), so their per-class scores are noisy, especially on the unseen-merchant split. Treat them with care.
- **Housing_Education** mixes home costs (property, cleaning, building materials, landscaping) with education, and most of its home-improvement transactions come from one contributor. Its unseen-person score depends heavily on who is held out.
- **Confidence and review (section 10):** report the threshold table (share auto-filed vs accuracy) and the reliability plot. They show the system knows when it doesn't know.
- The Khalifa data was labelled with the same sheet, so the external test measures a shift in bank, region and time, not in labeller.